# LangChain RAG with Ollama and Chroma

This notebook demonstrates a minimal Retrieval-Augmented Generation (RAG) pipeline using:

- LangChain for orchestration
- Ollama for the local LLM (and embeddings)
- Chroma as the vector store
- Local files in the `data/` directory as the knowledge base

Run the cells in order to ingest documents, build the vector store, construct the RAG chain, and ask questions.


## 1. Environment & configuration

This section loads configuration, checks that the data and Chroma directories exist, and performs a light connectivity check to Ollama.


In [1]:
from pathlib import Path

from rag.config import settings
from rag import ingest, chains

print("Data directory:", settings.data_dir.resolve())
print("Chroma directory:", settings.chroma_dir.resolve())
print("Ollama model:", settings.ollama_model)
print("Embedding model:", settings.embedding_model)

# Ensure directories exist
settings.data_dir.mkdir(parents=True, exist_ok=True)
settings.chroma_dir.mkdir(parents=True, exist_ok=True)

files = [p for p in settings.data_dir.rglob("*") if p.is_file()]
print(f"Found {len(files)} file(s) under {settings.data_dir}.")
for f in files[:10]:
    print(" -", f)

from langchain_ollama import ChatOllama

try:
    _test_llm = ChatOllama(model=settings.ollama_model)
    _ = _test_llm.invoke("Say 'ready' in one word.")
    print("Ollama is reachable and responding.")
except Exception as e:
    print("Warning: Ollama might not be running or the model is not available.")
    print("Details:", e)


Data directory: /Users/pedromuller/Developer/rag-langchain-python/data
Chroma directory: /Users/pedromuller/Developer/rag-langchain-python/chroma
Ollama model: llama3
Embedding model: nomic-embed-text
Found 1 file(s) under data.
 - data/sample.txt
Ollama is reachable and responding.


## 2. Ingestion / indexing

Load documents from `data/`, split them into chunks, and build (or rebuild) the Chroma vector store on disk.


In [2]:
from rag.ingest import load_documents, chunk_documents, build_vector_store
from rag.chains import get_embeddings

raw_docs = load_documents()
print(f"Loaded {len(raw_docs)} document(s).")

chunks = chunk_documents(raw_docs)
print(f"Split into {len(chunks)} chunk(s).")

embeddings = get_embeddings()
vector_store = build_vector_store(chunks, embeddings)
print("Vector store persisted to:", settings.chroma_dir.resolve())


Loaded 1 document(s).
Split into 1 chunk(s).
Vector store persisted to: /Users/pedromuller/Developer/rag-langchain-python/chroma


## 3. Build RAG chain

Create a retriever from the Chroma store and build a RetrievalQA-style RAG chain on top of the Ollama chat model.


In [3]:
from rag.ingest import get_retriever
from rag.chains import build_rag_chain, get_llm

retriever = get_retriever(embeddings)
llm = get_llm()
rag_chain = build_rag_chain(retriever, llm=llm)
rag_chain


<function rag.chains.build_rag_chain.<locals>._run(question: str)>

## 4. Ask questions

Helper function to query the RAG chain and optionally display the top source chunks.


In [4]:
from typing import Any, Dict

def ask(question: str, show_sources: bool = True) -> Dict[str, Any]:
    """Run a question through the RAG chain and pretty-print the answer."""
    result = rag_chain(question)
    answer = result.get("result")
    sources = result.get("source_documents", [])

    print("Answer:\n")
    print(answer)

    if show_sources:
        print("\nSources:")
        for i, doc in enumerate(sources, start=1):
            source = doc.metadata.get("source", "unknown")
            snippet = doc.page_content[:400].replace("\n", " ")
            if len(doc.page_content) > 400:
                snippet += "..."
            print(f"\n[{i}] {source}")
            print(snippet)

    return result

example_question = "What does the sample document say about RAG?"
ask(example_question)


Answer:

The sample document mentions that it is about LangChain-based Retrieval-Augmented Generation (RAG) using Ollama and Chroma. However, it does not provide any further information about what RAG is or how it works.

Sources:

[1] data/sample.txt
This is a sample document about LangChain-based Retrieval-Augmented Generation (RAG) using Ollama and Chroma.

[2] data/sample.txt
This is a sample document about LangChain-based Retrieval-Augmented Generation (RAG) using Ollama and Chroma.

[3] data/sample.txt
This is a sample document about LangChain-based Retrieval-Augmented Generation (RAG) using Ollama and Chroma.

[4] data/sample.txt
This is a sample document about LangChain-based Retrieval-Augmented Generation (RAG) using Ollama and Chroma.


{'result': 'The sample document mentions that it is about LangChain-based Retrieval-Augmented Generation (RAG) using Ollama and Chroma. However, it does not provide any further information about what RAG is or how it works.',
 'source_documents': [Document(id='aa970765-3a15-4fe8-866b-ff71873ee6dc', metadata={'source': 'data/sample.txt'}, page_content='This is a sample document about LangChain-based Retrieval-Augmented Generation (RAG) using Ollama and Chroma.'),
  Document(id='2dfa42bc-f98d-4c04-a3ff-be8a4dcdf7ae', metadata={'source': 'data/sample.txt'}, page_content='This is a sample document about LangChain-based Retrieval-Augmented Generation (RAG) using Ollama and Chroma.'),
  Document(id='86d3befb-31bd-4210-8818-23ed3f0958f8', metadata={'source': 'data/sample.txt'}, page_content='This is a sample document about LangChain-based Retrieval-Augmented Generation (RAG) using Ollama and Chroma.'),
  Document(id='b46e666d-0ad2-4ee3-a181-9b2693ebd192', metadata={'source': 'data/sample.txt'

## 5. Smoke tests

A couple of simple checks to ensure that retrieval and the RAG chain are wired correctly. These are intentionally lightweight.


In [5]:
# 1. Retriever smoke test
test_term = "Retrieval-Augmented Generation"
retrieved_docs = retriever.get_relevant_documents(test_term)
print(f"Retriever returned {len(retrieved_docs)} document chunk(s) for term {test_term!r}.")

# 2. RAG chain smoke test
smoke_question = "What is this project about?"
smoke_result = rag_chain(smoke_question)
print("\nSmoke test question:", smoke_question)
print("Smoke test answer:\n", smoke_result.get("result"))


Retriever returned 4 document chunk(s) for term 'Retrieval-Augmented Generation'.

Smoke test question: What is this project about?
Smoke test answer:
 Based on the provided context, this project appears to be about LangChain-based Retrieval-Augmented Generation (RAG) using Ollama and Chroma.
